In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchsummary import summary
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `t

In [5]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size ,patch_size ,num_hiddens):
        super().__init__()
        self.num_patches= (img_size// patch_size ) ** 2
        self.conv = nn.Conv2d(num_hiddens,kernel_size=patch_size,stride=patch_size)

    def forward(self,x):
        return self.conv(x).flatten(2).transpose(1,2)
    

class ViTMLP(nn.Module):
    def __init__(self,mlp_num_hidden, mlp_num_outputs,dropout=0.5):
        super().__init__()
        self.dense1=nn.LazyLinear(mlp_num_hidden)
        self.gelu = nn.GELU()
        self.dropout1=nn.Dropout(dropout)
        self.dense2=nn.LazyLinear(mlp_num_outputs)
        self.dropout2=nn.Dropout(dropout)

    def forward(self,x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(self.dense1(x)))))
    

class ViTBlock(nn.Module):
    def __init__(self,num_hidden,norm_shape,mlp_num_hiddens,num_heads,dropout,use_bias=False):
        super().__init__()
        self.ln1= nn.LayerNorm(norm_shape)
        self.attention = nn.MultiheadAttention(num_hidden,num_heads,dropout,use_bias)
        self.ln2= nn.LayerNorm(norm_shape)
        self.mlp= ViTMLP(mlp_num_hiddens,num_hidden,dropout)

    def forward(self,x):
        x2= self.ln1(x)
        attention_output,_=self.attention(x,x,x)
        x = x + attention_output
        x2= self.ln2(x)
        mlp_output= self.mlp(x2)
        x = x + mlp_output
        return x 
    
# class PositionalEncoding(nn.Module):
#     def __init__(self, d_model,dropout,max_length=1000):
#         super().__init__()
#         self.dropout = nn.Dropout(dropout)
#         encoding = torch.zeros(max_length, d_model)
#         position = torch.arange(0, max_length, dtype=torch.float).reshape(-1,1)
#         div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
#         encoding[:, 0::2] = torch.sin(position * div_term)
#         encoding[:, 1::2] = torch.cos(position * div_term)
#         encoding = encoding.unsqueeze(0)
#         self.register_buffer('encoding', encoding)
#     def forward(self, x):
#         return self.dropout(x + self.encoding[:, :x.size(1)].detach())
    


In [ ]:
class ViT(nn.Module):
    def __init__(self, eta, n_iter, random_state, batch_size, img_size, patch_size, num_hidden, norm_shape, mlp_num_hiddens, num_heads, num_blks, num_classes=100, dropout=0.2):
        super.init()
        self.eta = eta
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state

        self.patch_embeddding= PatchEmbedding(img_size,patch_size,num_hidden)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hidden))
        num_steps = self.patch_embedding.num_patches + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hidden))
        self.drop_out= nn.Dropout(dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(num_hidden,norm_shape,mlp_num_hiddens,num_heads,dropout,use_bias=False))
        self.lnorm= nn.LayerNorm(num_hidden)
        self.head=nn.Sequential(self.lnorm,nn.Linear(num_hidden, num_classes))
       
        self.device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        print(f"Using device: {self.device}")

        self.train_losses_     = []
        self.train_accuracies_ = []
        self.val_losses_       = []
        self.val_accuracies_   = []

        self.to(self.device)

    def forward(self,x):
        x = self.patch_embeddding(x)
        
        

        